# Step 10b: ZIP-level supermarket change maps (Fig. 9 panels)

Produces `zip_choropleth_two_panel_no_legend.png`, `legend_percent_change.png`, and `legend_symbols.png`, which `paper/scripts/10c_build_fig9.py` assembles into `Fig9_POIs_dynamics_map.pdf`.

Source: /glade/u/home/jcelis/POIs_filtered/AR_POIs_disruption_maps.ipynb, cells 1-34 (later cells are exploratory, raise errors when run in order, and overwrite the hourly-figure PNGs).

In [ ]:
# Paths (edit POI_DIR to point at a new step 9 run). Run this notebook from paper/notebooks/.
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
POI_DIR = "/glade/work/dgagne/cat_poi_rerun_20261008/supermarket_baseline_all_row_groups"  # step 9 output used in the paper
FFW_JSON = os.path.join(REPO_ROOT, "paper", "data", "ffw_LOX_20240328_20240331.json")  # cached IEM COW response
FIG_DIR = os.path.join(REPO_ROOT, "paper", "output", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
os.chdir(FIG_DIR)  # figures are saved here


# AR Supermarket Disruption Figures

Focused workflow for **Saturday 2024-03-30 vs baseline Saturdays**.

This notebook:
1. Loads the matched stays output
2. Merges POI names from the California POI GeoJSON
3. Prepares Saturday baseline vs storm-day subsets
4. Builds hourly visits and dwell figures
5. Computes store-level `% change in visits`, `% change in dwell`, and a combined disruption index
6. Makes two-panel and single-panel map figures with optional FFW and rain overlays

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from scipy.ndimage import gaussian_filter
import urllib3
import json
import xarray as xr

# ---------- PATHS ----------
base = POI_DIR
poi_path = "/glade/campaign/cisl/aiml/cat_flood/ancillary_data/POIs/California_filtered_pois.geojson"
rain_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/raw/mrms_precip_daily.nc"

storm_day = pd.Timestamp("2024-03-30")
counties = ["Los Angeles", "Ventura", "Santa Barbara"]
alpha = 0.5  # disruption index weight

print(base)
print(poi_path)
print(rain_path)
print("storm_day:", storm_day.date())

In [ ]:
# ---------- LOAD MATCHED VISITS + POI NAMES ----------
matched = pd.read_parquet(f"{base}/matched_stays_with_poi_latlon.parquet")
pois = gpd.read_file(poi_path)

poi_lookup = (
    pois[["id", "name", "type", "lat", "lon"]]
    .rename(columns={"id": "poi_id"})
    .drop_duplicates(subset=["poi_id"])
)

matched["poi_id"] = matched["poi_id"].astype(str)
poi_lookup["poi_id"] = poi_lookup["poi_id"].astype(str)

matched_named = matched.merge(poi_lookup, on="poi_id", how="left")

matched_named["datetime"] = pd.to_datetime(matched_named["datetime"], errors="coerce")
if getattr(matched_named["datetime"].dt, "tz", None) is not None:
    matched_named["datetime"] = matched_named["datetime"].dt.tz_localize(None)

matched_named["day"] = matched_named["datetime"].dt.floor("D")
matched_named["hour"] = matched_named["datetime"].dt.hour

print(matched_named[["poi_id", "name", "county", "datetime", "duration_min"]].head())
print("rows:", len(matched_named))
print("POIs with names:", matched_named["name"].notna().sum())

In [ ]:
# ---------- LOAD FFW POLYGONS + RAIN ----------
http = urllib3.PoolManager()
ffws = json.load(open(FFW_JSON))  # cached copy of the IEM COW API response

rain_ds = xr.open_dataset(rain_path)
print("FFW features:", len(ffws["events"]["features"]))
print("Rain max (in):", float(rain_ds["MultiSensor_QPE_01H_Pass2_00.00_sum"].sel(time="2024-03-30").max() / 25.4))

In [ ]:
# ---------- HELPERS ----------
def hourly_visits_by_county(df, county_name=None, storm_day=pd.Timestamp("2024-03-30"), weekday=5, smooth=True):
    d = df.copy()
    if county_name:
        d = d[d["county"] == county_name]

    subset = d[d["day"].dt.weekday == weekday].copy()
    storm = subset[subset["day"] == storm_day]
    baseline = subset[subset["day"] != storm_day]

    baseline_hour = (
        baseline.groupby("hour")["uid"].count() / max(baseline["day"].nunique(), 1)
    ).reindex(range(24), fill_value=0)

    storm_hour = (
        storm.groupby("hour")["uid"].count()
    ).reindex(range(24), fill_value=0)

    if smooth:
        baseline_hour = baseline_hour.rolling(3, center=True, min_periods=1).mean()
        storm_hour = storm_hour.rolling(3, center=True, min_periods=1).mean()

    return baseline_hour, storm_hour


def hourly_dwell_by_county(df, county_name=None, storm_day=pd.Timestamp("2024-03-30"), weekday=5, smooth=True):
    d = df.copy()
    if county_name:
        d = d[d["county"] == county_name]

    subset = d[d["day"].dt.weekday == weekday].copy()
    storm = subset[subset["day"] == storm_day]
    baseline = subset[subset["day"] != storm_day]

    baseline_hour = baseline.groupby("hour")["duration_min"].mean().reindex(range(24), fill_value=0)
    storm_hour = storm.groupby("hour")["duration_min"].mean().reindex(range(24), fill_value=0)

    if smooth:
        baseline_hour = baseline_hour.rolling(3, center=True, min_periods=1).mean()
        storm_hour = storm_hour.rolling(3, center=True, min_periods=1).mean()

    return baseline_hour, storm_hour


def plot_hourly_panels(df, metric="visits", storm_day=pd.Timestamp("2024-03-30"), weekday=5):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=False)
    panel_defs = [("All Counties", None)] + [(c, c) for c in counties]

    for ax, (title, county_name) in zip(axes.flatten(), panel_defs):
        if metric == "visits":
            b, s = hourly_visits_by_county(df, county_name, storm_day, weekday, smooth=True)
            ylabel = "Visits"
            panel_title = f"Hourly Supermarket Visits: Baseline vs Storm {'Saturday' if weekday == 5 else 'Friday'}"
        else:
            b, s = hourly_dwell_by_county(df, county_name, storm_day, weekday, smooth=True)
            ylabel = "Avg Dwell Time (min)"
            panel_title = f"Hourly Dwell Time: Baseline vs Storm {'Saturday' if weekday == 5 else 'Friday'}"

        label_base = f"Baseline {'Saturdays' if weekday == 5 else 'Fridays'}"
        label_storm = f"Storm {'Saturday' if weekday == 5 else 'Friday'}"

        ax.plot(b.index, b.values, color="steelblue", linewidth=3, marker="o", label=label_base)
        ax.plot(s.index, s.values, color="tomato", linewidth=3, marker="o", label=label_storm)

        ax.set_title(title, fontsize=13, weight="bold")
        ax.set_xticks(range(0, 24, 2))
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0, 0].legend(frameon=False)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel)
    for ax in axes[1, :]:
        ax.set_xlabel("Hour of Day")

    fig.suptitle(panel_title, fontsize=16, weight="bold")
    plt.tight_layout()
    plt.show()


def build_store_metrics(df, storm_day=pd.Timestamp("2024-03-30"), weekday=5):
    subset = df[df["day"].dt.weekday == weekday].copy()
    storm = subset[subset["day"] == storm_day].copy()
    baseline = subset[subset["day"] != storm_day].copy()

    n_baseline_days = baseline["day"].nunique()
    eps = 1e-6

    storm_store = (
        storm.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
        .agg(
            storm_visits=("uid", "count"),
            storm_dwell=("duration_min", "mean")
        )
    )

    baseline_store = (
        baseline.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
        .agg(
            baseline_total_visits=("uid", "count"),
            baseline_dwell=("duration_min", "mean")
        )
    )

    out = pd.concat([storm_store, baseline_store], axis=1).reset_index()
    out["baseline_visits"] = out["baseline_total_visits"] / max(n_baseline_days, 1)

    out["storm_visits"] = out["storm_visits"].fillna(0)
    out["storm_dwell"] = out["storm_dwell"].fillna(np.nan)
    out["baseline_visits"] = out["baseline_visits"].fillna(0)
    out["baseline_dwell"] = out["baseline_dwell"].fillna(np.nan)

    out["pct_change_visits"] = 100 * ((out["storm_visits"] - out["baseline_visits"]) / (out["baseline_visits"] + eps))
    out["pct_change_dwell"] = 100 * ((out["storm_dwell"] - out["baseline_dwell"]) / (out["baseline_dwell"] + eps))

    out["pct_change_visits_clip"] = out["pct_change_visits"].clip(-100, 100)
    out["pct_change_dwell_clip"] = out["pct_change_dwell"].clip(-100, 100)

    out["delta_visits"] = (out["storm_visits"] - out["baseline_visits"]) / (out["baseline_visits"] + eps)
    out["delta_dwell"] = (out["storm_dwell"] - out["baseline_dwell"]) / (out["baseline_dwell"] + eps)
    out["disruption_index"] = alpha * out["delta_visits"] + (1 - alpha) * out["delta_dwell"]
    out["disruption_index_clip"] = out["disruption_index"].clip(-1, 1)

    return out, n_baseline_days


def draw_overlays(ax):
    if "county_shapes" in globals():
        for county_shape in county_shapes:
            county_coords = np.array(county_shape.points).T
            ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.7, zorder=2)

    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(coords[:, 0], coords[:, 1],
                    facecolor="royalblue", edgecolor="royalblue",
                    alpha=0.12, linewidth=1.0, zorder=1)
        except Exception:
            pass

    rain_field = gaussian_filter(
        rain_ds["MultiSensor_QPE_01H_Pass2_00.00_sum"].sel(time="2024-03-30").values / 25.4, 1
    )
    rain_cont = ax.contour(
        rain_ds["lon"].values - 360,
        rain_ds["lat"].values,
        rain_field,
        levels=[1.0, 1.5, 2.0, 2.5, 3.0],
        cmap="plasma",
        linewidths=1.0,
        zorder=2
    )
    ax.clabel(rain_cont, fontsize=6)

    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.7")


## Hourly figures

In [ ]:
# Main hourly visits figure: baseline Saturdays vs storm Saturday
plot_hourly_panels(matched_named, metric="visits", storm_day=storm_day, weekday=5)

In [ ]:
# Supporting hourly dwell figure: baseline Saturdays vs storm Saturday
plot_hourly_panels(matched_named, metric="dwell", storm_day=storm_day, weekday=5)

## Store-level disruption metrics

In [ ]:
store_metrics, n_baseline_days = build_store_metrics(matched_named, storm_day=storm_day, weekday=5)
print("Baseline Saturdays:", n_baseline_days)
print(store_metrics[[
    "name", "county", "storm_visits", "baseline_visits",
    "storm_dwell", "baseline_dwell",
    "pct_change_visits", "pct_change_dwell", "disruption_index"
]].head(10))

## Two-panel map: % change in visits and dwell

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

metrics = [
    ("pct_change_visits_clip", "(a) % Change in Visits per Store"),
    ("pct_change_dwell_clip", "(b) % Change in Dwell Time per Store")
]

for ax, (metric, title) in zip(axes, metrics):
    cmap = plt.get_cmap("RdBu")   # red = decrease, blue = increase
    norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)

    sizes = 20 + 180 * (np.abs(store_metrics[metric].fillna(0)) / 100)

    ax.scatter(
        store_metrics["poi_lon"],
        store_metrics["poi_lat"],
        c=store_metrics[metric],
        s=sizes,
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.4,
        alpha=0.9,
        zorder=3
    )

    draw_overlays(ax)
    ax.set_title(title, fontsize=12, weight="bold")

cbar = fig.colorbar(
    ScalarMappable(norm=TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100), cmap=plt.get_cmap("RdBu")),
    ax=axes, fraction=0.03, pad=0.03
)
cbar.set_label("% Change\nRed = decrease, Blue = increase")

fig.suptitle(
    "Supermarket Change on 2024-03-30 relative to Baseline Saturdays\n"
    "Counties, Flash Flood Warnings, and MRMS Daily Rainfall",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 5), dpi=200)
ax = plt.gca()

cmap = plt.get_cmap("RdBu")
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sizes = 20 + 200 * np.abs(store_metrics["disruption_index_clip"].fillna(0))

sc = ax.scatter(
    store_metrics["poi_lon"],
    store_metrics["poi_lat"],
    c=store_metrics["disruption_index_clip"],
    s=sizes,
    cmap=cmap,
    norm=norm,
    edgecolor="k",
    linewidth=0.4,
    alpha=0.9,
    zorder=3
)

draw_overlays(ax)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Disruption Index\nRed = decrease, Blue = increase")

plt.title(
    "Store-Level Disruption Index on 2024-03-30\n"
    "α·ΔVisits + (1−α)·ΔDwell, with FFWs and MRMS Daily Rain",
    fontsize=13
)

plt.tight_layout()
plt.show()


In [ ]:
label_df = store_metrics.loc[np.abs(store_metrics["disruption_index_clip"]) >= 0.5].copy()

fig = plt.figure(figsize=(10, 5), dpi=200)
ax = plt.gca()

cmap = plt.get_cmap("RdBu")
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sizes = 20 + 200 * np.abs(store_metrics["disruption_index_clip"].fillna(0))

ax.scatter(
    store_metrics["poi_lon"],
    store_metrics["poi_lat"],
    c=store_metrics["disruption_index_clip"],
    s=sizes,
    cmap=cmap,
    norm=norm,
    edgecolor="k",
    linewidth=0.4,
    alpha=0.9,
    zorder=3
)

for _, row in label_df.iterrows():
    ax.text(row["poi_lon"] + 0.01, row["poi_lat"] + 0.01, str(row["name"]), fontsize=6, zorder=4)

draw_overlays(ax)
plt.title("Labeled Stores with Strongest Disruption Signal", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# If you want the Friday Santa Barbara comparison again, switch weekday=4 and storm_day to 2024-03-29.
friday_storm = pd.Timestamp("2024-03-29")
plot_hourly_panels(
    matched_named[matched_named["county"] == "Santa Barbara"].copy(),
    metric="visits",
    storm_day=friday_storm,
    weekday=4
)

## Store metrics (Map test)

In [ ]:
import pandas as pd
import numpy as np

storm_day = pd.Timestamp("2024-03-30")

matched_named["datetime"] = pd.to_datetime(matched_named["datetime"], errors="coerce")
if getattr(matched_named["datetime"].dt, "tz", None) is not None:
    matched_named["datetime"] = matched_named["datetime"].dt.tz_localize(None)

matched_named["day"] = matched_named["datetime"].dt.floor("D")

sat = matched_named[matched_named["day"].dt.weekday == 5].copy()

storm = sat[sat["day"] == storm_day].copy()
baseline = sat[sat["day"] != storm_day].copy()

n_baseline_days = baseline["day"].nunique()
print("Baseline Saturdays:", n_baseline_days)

storm_store = (
    storm.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
    .agg(
        storm_visits=("uid", "count"),
        storm_dwell=("duration_min", "mean")
    )
)

baseline_store = (
    baseline.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
    .agg(
        baseline_total_visits=("uid", "count"),
        baseline_dwell=("duration_min", "mean")
    )
)

store_metrics = pd.concat([storm_store, baseline_store], axis=1).reset_index()

store_metrics["baseline_visits"] = store_metrics["baseline_total_visits"] / n_baseline_days

store_metrics["storm_visits"] = store_metrics["storm_visits"].fillna(0)
store_metrics["storm_dwell"] = store_metrics["storm_dwell"].fillna(np.nan)
store_metrics["baseline_visits"] = store_metrics["baseline_visits"].fillna(0)
store_metrics["baseline_dwell"] = store_metrics["baseline_dwell"].fillna(np.nan)

eps = 1e-6

store_metrics["pct_change_visits"] = 100 * (
    (store_metrics["storm_visits"] - store_metrics["baseline_visits"]) /
    (store_metrics["baseline_visits"] + eps)
)

store_metrics["pct_change_dwell"] = 100 * (
    (store_metrics["storm_dwell"] - store_metrics["baseline_dwell"]) /
    (store_metrics["baseline_dwell"] + eps)
)

store_metrics["pct_change_visits_clip"] = store_metrics["pct_change_visits"].clip(-100, 100)
store_metrics["pct_change_dwell_clip"] = store_metrics["pct_change_dwell"].clip(-100, 100)

alpha = 0.5
store_metrics["delta_visits"] = (
    (store_metrics["storm_visits"] - store_metrics["baseline_visits"]) /
    (store_metrics["baseline_visits"] + eps)
)
store_metrics["delta_dwell"] = (
    (store_metrics["storm_dwell"] - store_metrics["baseline_dwell"]) /
    (store_metrics["baseline_dwell"] + eps)
)
store_metrics["disruption_index"] = (
    alpha * store_metrics["delta_visits"] +
    (1 - alpha) * store_metrics["delta_dwell"]
)
store_metrics["disruption_index_clip"] = store_metrics["disruption_index"].clip(-1, 1)

store_metrics.head()

In [ ]:
import json
import shapefile
import urllib3
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.path import Path

# county boundaries
county_shapes = shapefile.Reader(
    "/glade/campaign/cisl/aiml/cat_flood/ancillary_data/county_shapes/county_shapes"
).shapes()

# ZIP polygons
with open("/glade/work/cbecker/california_zip.geojson") as cal_zip_file:
    cal_zips = json.load(cal_zip_file)

zip_codes = np.array([f["properties"]["ZIP_CODE"] for f in cal_zips["features"]])

# FFW polygons
http = urllib3.PoolManager()
ffws = json.load(open(FFW_JSON))  # cached copy of the IEM COW API response

# rain
rain_ds = xr.open_dataset("/glade/campaign/cisl/aiml/cat_flood/mrms/raw/mrms_precip_daily.nc")

In [ ]:
def extract_polygons(geom):
    """
    Return a list of Nx2 arrays of lon/lat coordinates from a GeoJSON geometry.
    Supports Polygon and MultiPolygon.
    """
    if geom["type"] == "Polygon":
        return [np.array(geom["coordinates"][0])]
    elif geom["type"] == "MultiPolygon":
        return [np.array(poly[0]) for poly in geom["coordinates"]]
    else:
        return []

zip_paths = []
for feat in cal_zips["features"]:
    zcode = feat["properties"]["ZIP_CODE"]
    polys = extract_polygons(feat["geometry"])
    for poly in polys:
        zip_paths.append({
            "zip_code": zcode,
            "poly": poly,
            "path": Path(poly)
        })

def assign_zip_to_point(lon, lat, zip_paths):
    for item in zip_paths:
        if item["path"].contains_point((lon, lat)):
            return item["zip_code"]
    return None

store_metrics["zipcode"] = [
    assign_zip_to_point(lon, lat, zip_paths)
    for lon, lat in zip(store_metrics["poi_lon"], store_metrics["poi_lat"])
]

print(store_metrics["zipcode"].notna().sum(), "stores assigned to ZIP")
store_metrics[["name", "poi_lon", "poi_lat", "zipcode"]].head()

In [ ]:
zip_metrics = (
    store_metrics.dropna(subset=["zipcode"])
    .groupby("zipcode")
    .agg(
        visits_zip=("pct_change_visits_clip", "mean"),
        dwell_zip=("pct_change_dwell_clip", "mean"),
        disruption_zip=("disruption_index_clip", "mean"),
        n_stores=("poi_id", "count")
    )
)

zip_metrics.head()

In [ ]:
from matplotlib.colors import Normalize

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

metric_defs = [
    ("visits_zip", "(a) % Change in Visits by ZIP Code"),
    ("dwell_zip", "(b) % Change in Dwell Time by ZIP Code")
]

cmap = plt.get_cmap("RdBu")   # red = decrease, blue = increase
norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
sm = ScalarMappable(norm=norm, cmap=cmap)

for ax, (metric, title) in zip(axes, metric_defs):

    # fill ZIP polygons
    for feat in cal_zips["features"]:
        zcode = feat["properties"]["ZIP_CODE"]
        polys = extract_polygons(feat["geometry"])

        if zcode in zip_metrics.index:
            val = zip_metrics.loc[zcode, metric]
            color = sm.to_rgba(val)
        else:
            color = (0.92, 0.92, 0.92, 1)

        for poly in polys:
            ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

    # county boundaries
    for county_shape in county_shapes:
        county_coords = np.array(county_shape.points).T
        ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.8, zorder=3)

    # FFW polygons in green outline
    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
        except:
            pass

    # rain contours
    rain_field = rain_ds["MultiSensor_QPE_01H_Pass2_00.00_sum"].sel(time="2024-03-30").values / 25.4
    rain_cont = ax.contour(
        rain_ds["lon"].values - 360,
        rain_ds["lat"].values,
        rain_field,
        levels=[1.0, 1.5, 2.0, 2.5, 3.0],
        cmap="plasma",
        linewidths=1.0,
        zorder=2
    )
    ax.clabel(rain_cont, fontsize=6)

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.7")

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("% Change\nRed = decrease, Blue = increase")

fig.suptitle(
    "Supermarket Change on 2024-03-30 relative to Baseline Saturdays\n"
    "ZIP-level aggregation with FFWs (green) and MRMS daily rainfall",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# choose only stronger signals to avoid clutter
store_overlay = store_metrics.loc[
    (np.abs(store_metrics["pct_change_visits_clip"]) >= 50) |
    (np.abs(store_metrics["pct_change_dwell_clip"]) >= 50)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

for ax, (metric, title) in zip(axes, metric_defs):

    # ZIP fill
    for feat in cal_zips["features"]:
        zcode = feat["properties"]["ZIP_CODE"]
        polys = extract_polygons(feat["geometry"])

        if zcode in zip_metrics.index:
            val = zip_metrics.loc[zcode, metric]
            color = sm.to_rgba(val)
        else:
            color = (0.92, 0.92, 0.92, 1)

        for poly in polys:
            ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

    # triangles = store locations
    tri_metric = "pct_change_visits_clip" if metric == "visits_zip" else "pct_change_dwell_clip"
    ax.scatter(
        store_overlay["poi_lon"],
        store_overlay["poi_lat"],
        c=store_overlay[tri_metric],
        cmap=cmap,
        norm=norm,
        marker="^",
        s=35,
        edgecolor="k",
        linewidth=0.4,
        zorder=5
    )

    # county boundaries
    for county_shape in county_shapes:
        county_coords = np.array(county_shape.points).T
        ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.8, zorder=3)

    # FFW
    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
        except:
            pass

    # rain contours
    rain_field = rain_ds["MultiSensor_QPE_01H_Pass2_00.00_sum"].sel(time="2024-03-30").values / 25.4
    rain_cont = ax.contour(
        rain_ds["lon"].values - 360,
        rain_ds["lat"].values,
        rain_field,
        levels=[1.0, 1.5, 2.0, 2.5, 3.0],
        cmap="plasma",
        linewidths=1.0,
        zorder=2
    )
    ax.clabel(rain_cont, fontsize=6)

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.7")

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("% Change\nRed = decrease, Blue = increase")

fig.suptitle(
    "ZIP-level supermarket change with store overlays on 2024-03-30",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
cmap_di = plt.get_cmap("RdBu")
norm_di = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sm_di = ScalarMappable(norm=norm_di, cmap=cmap_di)

fig = plt.figure(figsize=(10, 5), dpi=200)
ax = plt.gca()

# ZIP fill
for feat in cal_zips["features"]:
    zcode = feat["properties"]["ZIP_CODE"]
    polys = extract_polygons(feat["geometry"])

    if zcode in zip_metrics.index:
        val = zip_metrics.loc[zcode, "disruption_zip"]
        color = sm_di.to_rgba(val)
    else:
        color = (0.92, 0.92, 0.92, 1)

    for poly in polys:
        ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

# optional small neutral store markers
ax.scatter(
    store_metrics["poi_lon"],
    store_metrics["poi_lat"],
    marker="^",
    s=15,
    color="k",
    alpha=0.5,
    zorder=5
)

# county boundaries
for county_shape in county_shapes:
    county_coords = np.array(county_shape.points).T
    ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.8, zorder=3)

# FFW in green
for ffw in ffws["events"]["features"]:
    try:
        coords = np.array(ffw["geometry"]["coordinates"][0][0])
        ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
    except:
        pass

# rain contours
rain_field = rain_ds["MultiSensor_QPE_01H_Pass2_00.00_sum"].sel(time="2024-03-30").values / 25.4
rain_cont = ax.contour(
    rain_ds["lon"].values - 360,
    rain_ds["lat"].values,
    rain_field,
    levels=[1.0, 1.5, 2.0, 2.5, 3.0],
    cmap="plasma",
    linewidths=1.0,
    zorder=2
)
ax.clabel(rain_cont, fontsize=6)

ax.set_xlim(-120.7, -117.4)
ax.set_ylim(33.65, 35.15)
ax.set_facecolor("0.7")

cbar = plt.colorbar(sm_di, ax=ax)
cbar.set_label("Disruption Index\nRed = decrease, Blue = increase")

plt.title(
    "ZIP-level Supermarket Disruption on 2024-03-30\n"
    "α·ΔVisits + (1−α)·ΔDwell, with FFWs (green) and MRMS rainfall",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

## Attempt 2

In [ ]:
import pandas as pd
import numpy as np

storm_day = pd.Timestamp("2024-03-30")

matched_named["datetime"] = pd.to_datetime(matched_named["datetime"], errors="coerce")
if getattr(matched_named["datetime"].dt, "tz", None) is not None:
    matched_named["datetime"] = matched_named["datetime"].dt.tz_localize(None)

matched_named["day"] = matched_named["datetime"].dt.floor("D")

sat = matched_named[matched_named["day"].dt.weekday == 5].copy()

storm = sat[sat["day"] == storm_day].copy()
baseline = sat[sat["day"] != storm_day].copy()

n_baseline_days = baseline["day"].nunique()

storm_store = (
    storm.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
    .agg(
        storm_visits=("uid", "count"),
        storm_dwell=("duration_min", "mean")
    )
)

baseline_store = (
    baseline.groupby(["poi_id", "name", "county", "poi_lat", "poi_lon"], dropna=False)
    .agg(
        baseline_total_visits=("uid", "count"),
        baseline_dwell=("duration_min", "mean")
    )
)

store_metrics = pd.concat([storm_store, baseline_store], axis=1).reset_index()

store_metrics["baseline_visits"] = store_metrics["baseline_total_visits"] / n_baseline_days

store_metrics["storm_visits"] = store_metrics["storm_visits"].fillna(0)
store_metrics["storm_dwell"] = store_metrics["storm_dwell"].fillna(np.nan)
store_metrics["baseline_visits"] = store_metrics["baseline_visits"].fillna(0)
store_metrics["baseline_dwell"] = store_metrics["baseline_dwell"].fillna(np.nan)

eps = 1e-6

store_metrics["pct_change_visits"] = 100 * (
    (store_metrics["storm_visits"] - store_metrics["baseline_visits"]) /
    (store_metrics["baseline_visits"] + eps)
)

store_metrics["pct_change_dwell"] = 100 * (
    (store_metrics["storm_dwell"] - store_metrics["baseline_dwell"]) /
    (store_metrics["baseline_dwell"] + eps)
)

store_metrics["pct_change_visits_clip"] = store_metrics["pct_change_visits"].clip(-100, 100)
store_metrics["pct_change_dwell_clip"] = store_metrics["pct_change_dwell"].clip(-100, 100)

alpha = 0.5
store_metrics["delta_visits"] = (
    (store_metrics["storm_visits"] - store_metrics["baseline_visits"]) /
    (store_metrics["baseline_visits"] + eps)
)
store_metrics["delta_dwell"] = (
    (store_metrics["storm_dwell"] - store_metrics["baseline_dwell"]) /
    (store_metrics["baseline_dwell"] + eps)
)
store_metrics["disruption_index"] = (
    alpha * store_metrics["delta_visits"] +
    (1 - alpha) * store_metrics["delta_dwell"]
)
store_metrics["disruption_index_clip"] = store_metrics["disruption_index"].clip(-1, 1)

In [ ]:
import json
import shapefile
import urllib3
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.path import Path

# 3-county boundary shapefile you already use
county_shapes = shapefile.Reader(
    "/glade/campaign/cisl/aiml/cat_flood/ancillary_data/county_shapes/county_shapes"
).shapes()

# ZIP polygons
with open("/glade/work/cbecker/california_zip.geojson") as f:
    cal_zips = json.load(f)

# Flash Flood Warning polygons
http = urllib3.PoolManager()
ffws = json.load(open(FFW_JSON))  # cached copy of the IEM COW API response

In [ ]:
import numpy as np

def extract_polygons(geom):
    if geom["type"] == "Polygon":
        return [np.array(geom["coordinates"][0])]
    elif geom["type"] == "MultiPolygon":
        return [np.array(poly[0]) for poly in geom["coordinates"]]
    return []

# build county paths
county_paths = []
for shp in county_shapes:
    pts = np.array(shp.points)
    if len(pts) > 2:
        county_paths.append(Path(pts))

def point_in_any_county(lon, lat, county_paths):
    return any(path.contains_point((lon, lat)) for path in county_paths)

# keep ZIPs whose representative point falls inside the 3-county region
study_zip_features = []

for feat in cal_zips["features"]:
    polys = extract_polygons(feat["geometry"])
    keep = False
    for poly in polys:
        cx, cy = poly[:, 0].mean(), poly[:, 1].mean()
        if point_in_any_county(cx, cy, county_paths):
            keep = True
            break
    if keep:
        study_zip_features.append(feat)

print("ZIPs retained:", len(study_zip_features))

In [ ]:
zip_paths = []
for feat in study_zip_features:
    zcode = feat["properties"]["ZIP_CODE"]
    polys = extract_polygons(feat["geometry"])
    for poly in polys:
        zip_paths.append({
            "zip_code": zcode,
            "path": Path(poly)
        })

def assign_zip_to_point(lon, lat, zip_paths):
    for item in zip_paths:
        if item["path"].contains_point((lon, lat)):
            return item["zip_code"]
    return None

store_metrics["zipcode"] = [
    assign_zip_to_point(lon, lat, zip_paths)
    for lon, lat in zip(store_metrics["poi_lon"], store_metrics["poi_lat"])
]

In [ ]:
zip_metrics = (
    store_metrics.dropna(subset=["zipcode"])
    .groupby("zipcode")
    .agg(
        visits_zip=("pct_change_visits_clip", "mean"),
        dwell_zip=("pct_change_dwell_clip", "mean"),
        disruption_zip=("disruption_index_clip", "mean"),
        n_stores=("poi_id", "count")
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

metric_defs = [
    ("visits_zip", "(a) % Change in Visits by ZIP Code"),
    ("dwell_zip", "(b) % Change in Dwell Time by ZIP Code")
]

cmap = plt.get_cmap("RdBu")   # red = decrease, blue = increase
norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
sm = ScalarMappable(norm=norm, cmap=cmap)

for ax, (metric, title) in zip(axes, metric_defs):

    # ZIP fill
    for feat in study_zip_features:
        zcode = feat["properties"]["ZIP_CODE"]
        polys = extract_polygons(feat["geometry"])

        if zcode in zip_metrics.index:
            val = zip_metrics.loc[zcode, metric]
            color = sm.to_rgba(val)
        else:
            color = (0.93, 0.93, 0.93, 1.0)

        for poly in polys:
            ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

    # county boundaries
    for county_shape in county_shapes:
        county_coords = np.array(county_shape.points).T
        ax.plot(
            county_coords[0], county_coords[1],
            color="k", linestyle="--", linewidth=0.8, zorder=3
        )

    # FFW polygons in green
    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(
                coords[:, 0], coords[:, 1],
                edgecolor="g", facecolor="none", lw=2, zorder=4
            )
        except:
            pass

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.85")

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("% Change\nRed = decrease, Blue = increase")

fig.suptitle(
    "Supermarket Change on 2024-03-30 relative to Baseline Saturdays\n"
    "ZIP-level aggregation with Flash Flood Warnings",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
store_overlay = store_metrics.loc[
    (np.abs(store_metrics["pct_change_visits_clip"]) >= 50) |
    (np.abs(store_metrics["pct_change_dwell_clip"]) >= 50)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

for ax, (metric, title) in zip(axes, metric_defs):

    for feat in study_zip_features:
        zcode = feat["properties"]["ZIP_CODE"]
        polys = extract_polygons(feat["geometry"])

        if zcode in zip_metrics.index:
            val = zip_metrics.loc[zcode, metric]
            color = sm.to_rgba(val)
        else:
            color = (0.93, 0.93, 0.93, 1.0)

        for poly in polys:
            ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

    tri_metric = "pct_change_visits_clip" if metric == "visits_zip" else "pct_change_dwell_clip"

    ax.scatter(
        store_overlay["poi_lon"],
        store_overlay["poi_lat"],
        c=store_overlay[tri_metric],
        cmap=cmap,
        norm=norm,
        marker="^",
        s=22,
        edgecolor="k",
        linewidth=0.3,
        zorder=5
    )

    for county_shape in county_shapes:
        county_coords = np.array(county_shape.points).T
        ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.8, zorder=3)

    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
        except:
            pass

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.85")

cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.03)
cbar.set_label("% Change\nRed = decrease, Blue = increase")

fig.suptitle(
    "ZIP-level supermarket change with selected store overlays\n"
    "2024-03-30 relative to baseline Saturdays",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
cmap_di = plt.get_cmap("RdBu")
norm_di = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sm_di = ScalarMappable(norm=norm_di, cmap=cmap_di)

fig = plt.figure(figsize=(10, 6), dpi=200)
ax = plt.gca()

for feat in study_zip_features:
    zcode = feat["properties"]["ZIP_CODE"]
    polys = extract_polygons(feat["geometry"])

    if zcode in zip_metrics.index:
        val = zip_metrics.loc[zcode, "disruption_zip"]
        color = sm_di.to_rgba(val)
    else:
        color = (0.93, 0.93, 0.93, 1.0)

    for poly in polys:
        ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

for county_shape in county_shapes:
    county_coords = np.array(county_shape.points).T
    ax.plot(county_coords[0], county_coords[1], color="k", linestyle="--", linewidth=0.8, zorder=3)

for ffw in ffws["events"]["features"]:
    try:
        coords = np.array(ffw["geometry"]["coordinates"][0][0])
        ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
    except:
        pass

ax.set_xlim(-120.7, -117.4)
ax.set_ylim(33.65, 35.15)
ax.set_facecolor("0.85")

cbar = plt.colorbar(sm_di, ax=ax)
cbar.set_label("Disruption Index\nRed = decrease, Blue = increase")

plt.title(
    "ZIP-level Supermarket Disruption on 2024-03-30\n"
    "Relative to Baseline Saturdays, with Flash Flood Warnings",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200, sharex=True, sharey=True)

metric_defs = [
    ("visits_zip", "(a) % Change in Visits by ZIP Code"),
    ("dwell_zip", "(b) % Change in Dwell Time by ZIP Code")
]

cmap = plt.get_cmap("RdBu")   # red = decrease, blue = increase
norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
sm = ScalarMappable(norm=norm, cmap=cmap)

for ax, (metric, title) in zip(axes, metric_defs):

    # ZIP fill
    for feat in study_zip_features:
        zcode = feat["properties"]["ZIP_CODE"]
        polys = extract_polygons(feat["geometry"])

        if zcode in zip_metrics.index:
            val = zip_metrics.loc[zcode, metric]
            color = sm.to_rgba(val)
        else:
            color = (0.93, 0.93, 0.93, 1.0)

        for poly in polys:
            ax.fill(poly[:, 0], poly[:, 1], color=color, ec="k", lw=0.4, zorder=1)

    # store triangles
    tri_metric = "pct_change_visits_clip" if metric == "visits_zip" else "pct_change_dwell_clip"

    ax.scatter(
        store_overlay["poi_lon"],
        store_overlay["poi_lat"],
        c=store_overlay[tri_metric],
        cmap=cmap,
        norm=norm,
        marker="^",
        s=22,
        edgecolor="k",
        linewidth=0.3,
        zorder=5
    )

    # county boundaries
    for county_shape in county_shapes:
        county_coords = np.array(county_shape.points).T
        ax.plot(
            county_coords[0], county_coords[1],
            color="k", linestyle="--", linewidth=0.8, zorder=3
        )

    # FFW polygons in green
    for ffw in ffws["events"]["features"]:
        try:
            coords = np.array(ffw["geometry"]["coordinates"][0][0])
            ax.fill(coords[:, 0], coords[:, 1], edgecolor="g", facecolor="none", lw=2, zorder=4)
        except:
            pass

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-120.7, -117.4)
    ax.set_ylim(33.65, 35.15)
    ax.set_facecolor("0.85")

fig.suptitle(
    "ZIP-level supermarket change with selected store overlays\n"
    "2024-03-30 relative to baseline Saturdays",
    fontsize=14, weight="bold"
)

plt.tight_layout()
plt.savefig("zip_choropleth_two_panel_no_legend.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2.2, 5), dpi=200)

cmap = plt.get_cmap("RdBu")
norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(sm, cax=ax)
cbar.set_label("% Change\nRed = decrease, Blue = increase", fontsize=11)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig("legend_percent_change.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(4, 2.2), dpi=200)
ax.axis("off")

legend_elements = [
    Patch(facecolor="lightgray", edgecolor="k", label="ZIP code"),
    Line2D([0], [0], marker="^", color="w", markerfacecolor="gray",
           markeredgecolor="k", markersize=8, label="Store"),
    Patch(facecolor="none", edgecolor="g", label="Flash Flood Warning")
]

ax.legend(handles=legend_elements, loc="center", frameon=False, fontsize=11)

plt.tight_layout()
plt.savefig("legend_symbols.png", bbox_inches="tight", dpi=300)
plt.show()